In [322]:
!pip install matplotlib
import math
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

Defaulting to user installation because normal site-packages is not writeable


# Foward and Backward Substitution


In [323]:
def FBSubstitution( L: np.array, U: np.array, pivots: np.array, B: np.array, pivoting: bool = False ) -> np.array:
    matrixSize = L.shape[0]
    y = np.zeros(matrixSize, dtype=float)

    B_permutated = np.zeros(matrixSize, dtype=float)
    if pivoting:
        for index in range(matrixSize):
            B_permutated[index] = B[pivots[index]]
    else:
        B_permutated = B.copy()

    for row in range(matrixSize):
        sum_L = 0.0
        for column in range(row):
            sum_L += L[row, column] * y[column]
        y[row] = (B_permutated[row] - sum_L) / L[row, row]

    vectorSolution = np.zeros(matrixSize, dtype=float)

    for row in range(matrixSize - 1, -1, -1):
        sum_U = 0.0
        for column in range(row + 1, matrixSize):
            sum_U += U[row, column] * vectorSolution[column]
        vectorSolution[row] = (y[row] - sum_U) / U[row, row]
    
    return vectorSolution

# LU Decomposition

In [356]:
def LuDecomposition( A: np.array ) -> tuple:
    A = np.array(A, dtype=float)  # copy, avoids mutating input
    matrixSize = A.shape[0]
    if A.shape[1] != matrixSize:
        raise ValueError("Matrix A must be square.")
    
    L = np.eye(matrixSize, dtype=float)
    U = np.zeros((matrixSize, matrixSize), dtype=float)
    pivot = np.zeros(matrixSize, dtype=int)
    for i in range(matrixSize):
        pivot[i] = i
    pDU = 1
    info = 0


    for activeColumn in range(matrixSize): # activeColumn == j

        lagestValue = activeColumn # p
        for inspector in range(activeColumn + 1, matrixSize): # inspector == k
            if abs(A[inspector, activeColumn]) > abs(A[lagestValue, activeColumn]):
                lagestValue = inspector

        if activeColumn != lagestValue:
            A[[activeColumn, lagestValue], :] = A[[lagestValue, activeColumn], :]
            
            temp_p = pivot[activeColumn]
            pivot[activeColumn] = pivot[lagestValue]
            pivot[lagestValue] = temp_p
            
            pDU = -pDU
        
        pDU *= A[activeColumn, activeColumn]

        if A[activeColumn, activeColumn] != 0:
            ratio = 1.0 / A[activeColumn, activeColumn]
            for row in range(activeColumn + 1, matrixSize):
                mult = A[row, activeColumn] * ratio
                A[row, activeColumn] = mult
                for column in range(activeColumn + 1, matrixSize):
                    A[row, column] -= mult * A[activeColumn, column]
        else:
            if info == 0:
                info = activeColumn + 1
            
            
    for row in range(matrixSize):
        for column in range(matrixSize):
            if row > column:
                L[row, column] = A[row, column]
            else:
                U[row, column] = A[row, column]

    return A, L, U, pivot, pDU, info


matrixInput = np.array([[0, 1, -2], [2, -2, 3], [1, 3, 1]], dtype=float)

A, L, U, pivot, pDU, info = LuDecomposition(matrixInput)

print("A:\n", A)
print("L:\n", L)
print("U:\n", U)
print("Pivot:\n", pivot)

result = FBSubstitution(L, U, pivot, np.array([7, -10, 8], dtype=float), pivoting=True)

print("Vector Solution:\n", result)

"""
Wanted Result

A:
 [[ 2.    -2.     3.   ]
 [ 0.5    4.    -0.5  ]
 [ 0.     0.25  -1.875]]
L:
 [[1.   0.   0.  ]
 [0.5  1.   0.  ]
 [0.   0.25 1.  ]]
U:
 [[ 2.    -2.     3.   ]
 [ 0.     4.    -0.5  ]
 [ 0.     0.    -1.875]]
Pivot:
 [1 2 0] 
Vector Solution:
 [ 1.  3. -2.]
"""


A:
 [[ 2.    -2.     3.   ]
 [ 0.5    4.    -0.5  ]
 [ 0.     0.25  -1.875]]
L:
 [[1.   0.   0.  ]
 [0.5  1.   0.  ]
 [0.   0.25 1.  ]]
U:
 [[ 2.    -2.     3.   ]
 [ 0.     4.    -0.5  ]
 [ 0.     0.    -1.875]]
Pivot:
 [1 2 0]
Vector Solution:
 [ 1.  3. -2.]


'\nWanted Result\n\nA:\n [[ 2.    -2.     3.   ]\n [ 0.5    4.    -0.5  ]\n [ 0.     0.25  -1.875]]\nL:\n [[1.   0.   0.  ]\n [0.5  1.   0.  ]\n [0.   0.25 1.  ]]\nU:\n [[ 2.    -2.     3.   ]\n [ 0.     4.    -0.5  ]\n [ 0.     0.    -1.875]]\nPivot:\n [1 2 0] \nVector Solution:\n [ 1.  3. -2.]\n'

# Cholesky Decomposition

In [357]:
def CholeskyDecomposition( A: np.array ) -> tuple:
    A = np.array(A, dtype=float)  # copy, avoids mutating input
    matrixSize = A.shape[0]

    determinant = 1.0
    info = 0

    if A.shape[1] != matrixSize:
        raise ValueError("Matrix A must be square.")

    for j in range(matrixSize):
        sums = 0
        for k in range(j):
            sums += A[j, k] ** 2
        temp = A[j, j] - sums
        if temp > 0:
            A[j, j] = math.sqrt(temp)
            r = 1.0 / A[j, j]
            determinant *= temp
        else:
            info = j + 1
            print("Matrix A is not positive definite.")
            return A, np.tril(A), determinant, info
        for i in range(j + 1, matrixSize):
            sums = 0
            for k in range(j):
                sums = sums + A[i, k] * A[j, k]
            A[i, j] = (A[i, j] - sums) * r

    return A, np.tril(A), determinant, info



matrixInput = np.array([[4, 12, -16], [12, 37, -43], [-16, -43, 98]], dtype=float)

A, L, determinant, info = CholeskyDecomposition(matrixInput)

b = np.array([-20, -43, 192])

vectorsolution = FBSubstitution(L, L.T, None, b, pivoting=False)

print("A:\n", A)
print("L:\n", L)
print("Vector Solution:\n", vectorsolution)
print("Determinant:\n", determinant)
print("Info:\n", info)


""" 
Wanted results

L = [[ 2, 0, 0],
     [ 6, 1, 0],
     [-8, 5, 3]]
vectorsolution = [1, 2, 3]
determinant = 36

"""


A:
 [[  2.  12. -16.]
 [  6.   1. -43.]
 [ -8.   5.   3.]]
L:
 [[ 2.  0.  0.]
 [ 6.  1.  0.]
 [-8.  5.  3.]]
Vector Solution:
 [1. 2. 3.]
Determinant:
 36.0
Info:
 0


' \nWanted results\n\nL = [[ 2, 0, 0],\n     [ 6, 1, 0],\n     [-8, 5, 3]]\nvectorsolution = [1, 2, 3]\ndeterminant = 36\n\n'

# Gauss-Jacobi 

In [326]:
def GaussJacobi(A: np.array, b: np.array, maxIterations: int, precision: float, xo:np.array = None, usingStartArray:bool = False) -> np.array:
    matrixSize = A.shape[0]
    v = np.zeros(matrixSize, dtype=float)
    vectorSolution = np.zeros(matrixSize, dtype=float)
    iterations = 0
    info = 0


    if A.shape[1] != matrixSize:
        raise ValueError("Matrix A must be square.")
    
    if not usingStartArray:
        for i in range(matrixSize):
            vectorSolution[i] = b[i] / A[i, i]
    else:
        vectorSolution = xo.copy()


    while True:
        iterations += 1
        for i in range(matrixSize):
            sumAx = 0.0
            for j in range(matrixSize):
                if i != j:
                    sumAx += A[i, j] * vectorSolution[j]
            v[i] = (b[i] - sumAx) / A[i, i]
        normaNum = 0.0
        normoDen = 0.0

        for i in range(matrixSize):
            t = abs(v[i] - vectorSolution[i])
            if t > normaNum:
                normaNum = t
            if abs(v[i]) > normoDen:
                normoDen = abs(v[i])
            vectorSolution[i] = v[i] 
        normaRel = normaNum / normoDen

        print(f"Iteration: {iterations}, Norma Relative: {normaRel:.5e}, Vector Solution: {vectorSolution}")

        if (iterations >= maxIterations) or (normaRel <= precision):
            if iterations >= maxIterations:
                info = 1
            else:
                info = 0
            break
    
    return vectorSolution, iterations, info




A = np.array([[4, 1, 1], [1, 5, 2], [1, 2, 6]], dtype=float)
b = np.array([9, 17, 23], dtype=float)
Toler = 1e-20
IterMax = 200

vectorsolution, iters, info = GaussJacobi(A, b, IterMax, Toler)

print("Vector Solution:\n", vectorsolution)


# Result = [ 1 , 2 , 3 ]

Iteration: 1, Norma Relative: 8.53047e-01, Vector Solution: [0.44166667 1.41666667 2.325     ]
Iteration: 2, Norma Relative: 2.93536e-01, Vector Solution: [1.31458333 2.38166667 3.2875    ]
Iteration: 3, Norma Relative: 1.98409e-01, Vector Solution: [0.83270833 1.82208333 2.82034722]
Iteration: 4, Norma Relative: 9.17457e-02, Vector Solution: [1.08939236 2.10531944 3.0871875 ]
Iteration: 5, Norma Relative: 5.35841e-02, Vector Solution: [0.95187326 1.94724653 2.94999479]
Iteration: 6, Norma Relative: 2.72279e-02, Vector Solution: [1.02568967 2.02962743 3.02560561]
Iteration: 7, Norma Relative: 1.50737e-02, Vector Solution: [0.98619174 1.98461982 2.98584258]
Iteration: 8, Norma Relative: 7.91533e-03, Vector Solution: [1.0073844  2.00842462 3.0074281 ]
Iteration: 9, Norma Relative: 4.29670e-03, Vector Solution: [0.99603682 1.99555188 2.99596106]
Iteration: 10, Norma Relative: 2.28381e-03, Vector Solution: [1.00212177 2.00240821 3.00214324]
Iteration: 11, Norma Relative: 1.23043e-03, Vecto

# Gauss-Seidel

In [327]:
def GaussSeidel(A: np.array, b: np.array, maxIterations: int, precision: float, xo:np.array = None, usingStartArray:bool = False) -> np.array:
    matrixSize = A.shape[0]
    v = np.zeros(matrixSize, dtype=float)
    vectorSolution = np.zeros(matrixSize, dtype=float)
    iterations = 0
    info = 0


    if A.shape[1] != matrixSize:
        raise ValueError("Matrix A must be square.")

    if not usingStartArray:
        for i in range(matrixSize):
            vectorSolution[i] = b[i] / A[i, i]
    else:
        vectorSolution = xo.copy()

    while True:
        iterations += 1
        normaNum = 0.0
        normoDen = 0.0
        for i in range(matrixSize):
            sumAx = 0.0
            for j in range(matrixSize):
                if i != j:
                    sumAx += A[i, j] * vectorSolution[j]
            v[i] = vectorSolution[i]
            vectorSolution[i] = (b[i] - sumAx) / A[i, i]
            t = abs(v[i] - vectorSolution[i])
            if t > normaNum:
                normaNum = t
            if abs(vectorSolution[i]) > normoDen:
                normoDen = abs(vectorSolution[i])
        normaRel = normaNum / normoDen

        print(f"Iteration: {iterations}, Norma Relative: {normaRel:.5e}, Vector Solution: {vectorSolution}")

        if (iterations >= maxIterations) or (normaRel <= precision):
            if iterations >= maxIterations:
                info = 1
            else:
                info = 0
            break
    return vectorSolution, iterations, info




A = np.array([[4, 1, 1], [1, 5, 2], [1, 2, 6]], dtype=float)
b = np.array([9, 17, 23], dtype=float)
Toler = 1e-20
IterMax = 200

vectorsolution, iters, info = GaussSeidel(A, b, IterMax, Toler)

print("Vector Solution:\n", vectorsolution)


# Result = [ 1 , 2 , 3 ]

Iteration: 1, Norma Relative: 5.71003e-01, Vector Solution: [0.44166667 1.77833333 3.16694444]
Iteration: 2, Norma Relative: 1.89353e-01, Vector Solution: [1.01368056 1.93048611 3.0208912 ]
Iteration: 3, Norma Relative: 1.95652e-02, Vector Solution: [1.01215567 1.98921238 3.00156993]
Iteration: 4, Norma Relative: 3.28377e-03, Vector Solution: [1.00230442 1.99891114 2.99997888]
Iteration: 5, Norma Relative: 6.75650e-04, Vector Solution: [1.00027749 1.99995295 2.99996943]
Iteration: 6, Norma Relative: 8.60300e-05, Vector Solution: [1.0000194  2.00000835 2.99999398]
Iteration: 7, Norma Relative: 6.66216e-06, Vector Solution: [0.99999942 2.00000252 2.99999926]
Iteration: 8, Norma Relative: 7.12096e-07, Vector Solution: [0.99999956 2.00000039 2.99999995]
Iteration: 9, Norma Relative: 1.20594e-07, Vector Solution: [0.99999992 2.00000004 3.        ]
Iteration: 10, Norma Relative: 2.43592e-08, Vector Solution: [0.99999999 2.         3.        ]
Iteration: 11, Norma Relative: 3.06492e-09, Vecto

# Newton

In [358]:
def Newton( F: callable , J: callable , x0: np.array , precision: float = 1e-10 , maxIterations: int = 100 ) -> tuple:
    vectorSolution = np.array(x0, dtype=float)
    iterations = 0
    info = 0
    while True:
        Jx = J(vectorSolution)
        Fx = F(vectorSolution)

        if np.linalg.norm(Fx, np.inf) <= precision:
            info = 0
            break

        try:
            s = np.linalg.solve(Jx, -Fx)
        except np.linalg.LinAlgError:
            info = 2
            break
        vectorSolution = vectorSolution + s

        iterations += 1
        
        print(f"Iteration: {iterations}, Vector Solution: {vectorSolution}, ||F(x)||_inf: {np.linalg.norm(Fx, np.inf):.5e}")

        if np.linalg.norm(s, np.inf) <= precision:
            info = 0
            break

        
        if iterations >= maxIterations:
            info = 1
            break
    return vectorSolution, iterations, info

F = lambda x: np.array([3*x[0] - np.cos(x[1]*x[2]) - 0.5, x[0]**2 - 81*(x[1]+0.1)**2 + np.sin(x[2]) + 1.06, np.exp(-x[0]*x[1]) + 20*x[2] + (10*np.pi - 3)/3])

J = lambda x: np.array([[3, x[2]*np.sin(x[1]*x[2]), x[1]*np.sin(x[1]*x[2])], [2*x[0], -162*(x[1]+0.1), np.cos(x[2])], [-x[1]*np.exp(-x[0]*x[1]), -x[0]*np.exp(-x[0]*x[1]), 20]])

vectorSolution, iterations, info = Newton(F, J, np.array([0.1, 0.1, -0.1]), precision=1e-10, maxIterations=100)

print("Vector Solution:\n", vectorSolution)

        

Iteration: 1, Vector Solution: [ 0.49986967  0.01946685 -0.52152047], ||F(x)||_inf: 8.46203e+00
Iteration: 2, Vector Solution: [ 0.50001424  0.00158859 -0.52355696], ||F(x)||_inf: 3.44388e-01
Iteration: 3, Vector Solution: [ 5.00000113e-01  1.24447833e-05 -5.23598450e-01], ||F(x)||_inf: 2.58891e-02
Iteration: 4, Vector Solution: [ 5.00000000e-01  7.75785716e-10 -5.23598776e-01], ||F(x)||_inf: 2.01223e-04
Iteration: 5, Vector Solution: [ 5.00000000e-01  3.35521889e-18 -5.23598776e-01], ||F(x)||_inf: 1.25431e-08
Vector Solution:
 [ 5.00000000e-01  3.35521889e-18 -5.23598776e-01]


# Modified Newton

In [329]:
def ModifiedNewton( F: callable , J: callable , x0: np.array , precision: float = 1e-10 , maxIterations: int = 100 ) -> tuple:
    vectorSolution = np.array(x0, dtype=float)
    iterations = 0
    info = 0
    Jx0 = J(vectorSolution)

    while True:
        Fx = F(vectorSolution)

        if np.linalg.norm(Fx, np.inf) <= precision:
            info = 0
            break

        try:
            s = np.linalg.solve(Jx0, -Fx)
        except np.linalg.LinAlgError:
            info = 2
            break
        vectorSolution = vectorSolution + s

        iterations += 1
        
        print(f"Iteration: {iterations}, Vector Solution: {vectorSolution}, ||F(x)||_inf: {np.linalg.norm(Fx, np.inf):.5e}")

        if np.linalg.norm(s, np.inf) <= precision:
            info = 0
            break

        
        if iterations >= maxIterations:
            info = 1
            break
    return vectorSolution, iterations, info

F = lambda x: np.array([3*x[0] - np.cos(x[1]*x[2]) - 0.5, x[0]**2 - 81*(x[1]+0.1)**2 + np.sin(x[2]) + 1.06, np.exp(-x[0]*x[1]) + 20*x[2] + (10*np.pi - 3)/3])

J = lambda x: np.array([[3, x[2]*np.sin(x[1]*x[2]), x[1]*np.sin(x[1]*x[2])], [2*x[0], -162*(x[1]+0.1), np.cos(x[2])], [-x[1]*np.exp(-x[0]*x[1]), -x[0]*np.exp(-x[0]*x[1]), 20]])

vectorSolution, iterations, info = ModifiedNewton(F, J, np.array([0.1, 0.1, -0.1]), precision=1e-10, maxIterations=100)

print("Vector Solution:\n", vectorSolution)

Iteration: 1, Vector Solution: [ 0.49986967  0.01946685 -0.52152047], ||F(x)||_inf: 8.46203e+00
Iteration: 2, Vector Solution: [ 0.49998583  0.00878775 -0.52316688], ||F(x)||_inf: 3.44388e-01
Iteration: 3, Vector Solution: [ 0.49999793  0.00420477 -0.5234022 ], ||F(x)||_inf: 1.48257e-01
Iteration: 4, Vector Solution: [ 0.49999987  0.00206025 -0.52350437], ||F(x)||_inf: 6.93812e-02
Iteration: 5, Vector Solution: [ 0.50000014  0.00102056 -0.52355244], ||F(x)||_inf: 3.36382e-02
Iteration: 6, Vector Solution: [ 5.00000115e-01  5.08200199e-04 -5.23575805e-01], ||F(x)||_inf: 1.65771e-02
Iteration: 7, Vector Solution: [ 5.00000069e-01  2.53717695e-04 -5.23587332e-01], ||F(x)||_inf: 8.23375e-03
Iteration: 8, Vector Solution: [ 5.00000037e-01  1.26829787e-04 -5.23593061e-01], ||F(x)||_inf: 4.10546e-03
Iteration: 9, Vector Solution: [ 5.00000019e-01  6.34407076e-05 -5.23595919e-01], ||F(x)||_inf: 2.05096e-03
Iteration: 10, Vector Solution: [ 5.00000010e-01  3.17433466e-05 -5.23597347e-01], ||F(x

# Relatório 2

## 1 – Transferência de calor

Uma discretização de um problema estacionário de transferência de calor conduz ao sistema $A\,\mathbf{T} = b$ com

$$
A = \begin{bmatrix}
7 & -2 & -1 & 0 \\
-2 & 6 & -1 & -1 \\
-1 & -1 & 7 & -2 \\
0 & -1 & -2 & 7
\end{bmatrix},
\qquad
b = \begin{bmatrix} 400 \\ 40 \\ 240 \\ 80 \end{bmatrix}.
$$

As incógnitas $T_i$ ($T_1, T_2, T_3, T_4$, em °C) representam temperaturas nodais.

### a) verificar se $A$ é simétrica;

### Answer

The $A$ matrix is square ans symetric, checked by checking the number of rows and columns to check if it is square, and after transposed the matrix and check using a for if it was equal to the original matrix.

In [330]:
A = np.array([[7, -2, -1, 0], [-2, 6, -1, -1], [-1, -1, 7, -2], [0, -1, -2, 7]], dtype=float)
b = np.array([400, 40, 240, 80], dtype=float)
ATransposed = A.T

rows = A.shape[0]
columns = A.shape[1]


if rows == columns:
    print("Matrix is Square\n")
else:
    print("Matrix is Not Square\n")
print(f"Number of Rows: {rows} | Number of Columns: {columns}\n")
print(f"A Matrix: \n {A}\n")
print(f"A Matrix Transposed: \n {ATransposed}\n")

flag = True

for row in range(rows):
    for column in range(columns):
        if A[row][column] != ATransposed[row][column]:
            flag = False
        
if flag:
    print("Matrix is Symmetric\n")
else:
    print("Matrix is Not Symmetric\n")


Matrix is Square

Number of Rows: 4 | Number of Columns: 4

A Matrix: 
 [[ 7. -2. -1.  0.]
 [-2.  6. -1. -1.]
 [-1. -1.  7. -2.]
 [ 0. -1. -2.  7.]]

A Matrix Transposed: 
 [[ 7. -2. -1.  0.]
 [-2.  6. -1. -1.]
 [-1. -1.  7. -2.]
 [ 0. -1. -2.  7.]]

Matrix is Symmetric



### b) verificar a dominância diagonal;

### Answer

It is diagonally dominant checked by the code bellow that sums the rows without the diagonals, in the output we can see that the value of the diagonal is bigger than the sum of it's row, so it is diagonally dominant.

In [331]:
for row in range(rows):
    diagValue = abs(A[row][row])
    rowValue = 0
    for column in range(columns):
        if column != row:
            rowValue += abs(A[row][column])
    print( f"A[{row}][{row}] Value = {diagValue}\nRow Sum Value: {rowValue}\n" )
    if diagValue <= rowValue:
        print( "Matrix Not Diagonally Dominant\n" )


A[0][0] Value = 7.0
Row Sum Value: 3.0

A[1][1] Value = 6.0
Row Sum Value: 4.0

A[2][2] Value = 7.0
Row Sum Value: 4.0

A[3][3] Value = 7.0
Row Sum Value: 3.0



### c) calcular os autovalores e verificar se $A$ é definida positiva;

### Answer

The Eigenvalues are [3.11985506, 6.22683301, 8.24642152, 9.40689042], all of them are positive so the matrix is positive defined.

In [332]:
eigenValues = np.linalg.eigvalsh(A)
definedPositive = np.all(eigenValues > 0)

print(f"EigenValues: {eigenValues}")
print(f"It is Positive Defined? {definedPositive}")

EigenValues: [3.11985506 6.22683301 8.24642152 9.40689042]
It is Positive Defined? True


### d) resolver o sistema usando Fatoração LU;

### Answer

For Lu Decompositon the A matrix must be square, that it is for the testing above, the leading principals are non zero checking the actual matrix we can see that, be non singular that we know because the info return on the Lu decomposition was 0, so the result can be trusted, the result found was [80.55739881, 50.2189781, 63.46383543, 36.73523557].



In [333]:
Alu, L, U, pivot, pDU, info = LuDecomposition( A )
vectorSolutionLU = FBSubstitution( L, U , pivot, b, pivoting=True )

print(f"info = {info}\n")
print(f"L:\n{L}\n")
print(f"U:\n{U}\n")
print(f"Solution = {vectorSolutionLU}\n")


info = 0

L:
[[ 1.          0.          0.          0.        ]
 [-0.28571429  1.          0.          0.        ]
 [-0.14285714 -0.23684211  1.          0.        ]
 [ 0.         -0.18421053 -0.34136546  1.        ]]

U:
[[ 7.         -2.         -1.          0.        ]
 [ 0.          5.42857143 -1.28571429 -1.        ]
 [ 0.          0.          6.55263158 -2.23684211]
 [ 0.          0.          0.          6.05220884]]

Solution = [80.55739881 50.2189781  63.46383543 36.73523557]



### e) resolver o sistema usando Fatoração Cholesky;

### Answer

To use Cholesky Decompositon is need that the matrix is square, symmetric, and that is positive defined, things we tested in previus questions.

The vector solution was the same was the other method as expected, [80.55739881, 50.2189781, 63.46383543, 36.73523557]

In [334]:
ACho, L, determinant, info = CholeskyDecomposition(A)

vectorSolutionCholesky = FBSubstitution(L, L.T, None, b, pivoting=False)


print(f"info = {info}\n")
print(f"L:\n{L}\n")
print(f"LT:\n{L.T}\n")
print(f"Solution = {vectorSolutionCholesky}\n")

info = 0

L:
[[ 2.64575131  0.          0.          0.        ]
 [-0.75592895  2.32992949  0.          0.        ]
 [-0.37796447 -0.55182541  2.55981085  0.        ]
 [ 0.         -0.42919754 -0.87383101  2.46012374]]

LT:
[[ 2.64575131 -0.75592895 -0.37796447  0.        ]
 [ 0.          2.32992949 -0.55182541 -0.42919754]
 [ 0.          0.          2.55981085 -0.87383101]
 [ 0.          0.          0.          2.46012374]]

Solution = [80.55739881 50.2189781  63.46383543 36.73523557]



### f) calcular $\lVert b - Ax \rVert_\infty$ para ambas as soluções;

### Answer

LU residual: 4.26326e-14

Cholesky residual: 5.68434e-14

In [335]:
residualLU = np.linalg.norm(b - A @ vectorSolutionLU, np.inf)
residualCholesky = np.linalg.norm(b - A @ vectorSolutionCholesky, np.inf)

print(f"LU residual: {residualLU:.5e}")
print(f"Cholesky residual: {residualCholesky:.5e}")

LU residual: 4.26326e-14
Cholesky residual: 5.68434e-14


### g) interpretar como a modificação de $b$ para $b_{novo}$ afeta as temperaturas.

$$b_{novo} = \begin{bmatrix} 480 \\ 40 \\ 240 \\ 80 \end{bmatrix}.$$

### Answer

Raising only the first entry of `b` from 400 to 480 (+80) increases all four temperatures. Since the system is linear, `ΔT = A⁻¹ Δb` with `Δb = [80, 0, 0, 0]`. LU and Cholesky give the same new solution.

| Node | Before (°C) | After (°C) | Change (°C) |
|---|---|---|---|
| T₁ | 80.557 | 93.935 | +13.378 |
| T₂ | 50.219 | 55.474 | +5.255 |
| T₃ | 63.464 | 66.596 | +3.132 |
| T₄ | 36.735 | 38.381 | +1.646 |

All temperatures rise, because `A` is symmetric, diagonally dominant and has negative off-diagonal entries, so `A⁻¹` has no negative entries. The increase is largest at T₁ (where `b` changed) and falls with the coupling to it: T₂ (`a₁₂ = −2`) > T₃ (`a₁₃ = −1`) > T₄ (`a₁₄ = 0`, only indirect).

In [336]:
bNovo = np.array([480, 40, 240, 80], dtype=float)

Alu, L, U, pivot, pDU, info = LuDecomposition( A )
vectorSolutionLUNovo = FBSubstitution( L, U , pivot, bNovo, pivoting=True )

ACho, L, determinant, info = CholeskyDecomposition(A)
vectorSolutionCholeskyNovo = FBSubstitution(L, L.T, None, bNovo, pivoting=False)


print(f"LU Solution: {vectorSolutionLUNovo}\n")
print(f"Cholesky Solution: {vectorSolutionCholeskyNovo}\n")

LU Solution: [93.93497014 55.47445255 66.59588587 38.38088918]

Cholesky Solution: [93.93497014 55.47445255 66.59588587 38.38088918]



## Questão 2 – Rede de correntes

O balanço de correntes de uma rede resulta em $A\,\mathbf{V} = b$ com

$$
A = \begin{bmatrix}
10 & -2 & -1 & 0 & 0 \\
-2 & 9 & -2 & -1 & 0 \\
-1 & -2 & 10 & -2 & -1 \\
0 & -1 & -2 & 9 & -2 \\
0 & 0 & -1 & -2 & 8
\end{bmatrix},
\qquad
b = \begin{bmatrix} 12 \\ 5 \\ 0 \\ 5 \\ 10 \end{bmatrix}.
$$

As incógnitas correspondem às tensões nodais $V_1, \dots, V_5$. Use

$$x^{(0)} = \begin{bmatrix} 0 \\ 0 \\ 0 \\ 0 \\ 0 \end{bmatrix}.$$

In [337]:
Aq2 = np.array([[10, -2,-1 ,0 , 0], [-2, 9, -2, -1, 0], [-1, -2, 10, -2, -1], [0, -1, -2, 9, -2], [0, 0, -1, -2, 8]], dtype=float)
bq2 = np.array([12, 5, 0, 5 , 10], dtype=float)
xoq2 = np.array([0,0,0,0,0], dtype=float)
rowsq2 = Aq2.shape[0]
columnsq2 = Aq2.shape[1]

print(f"Rows = {rowsq2} | Columns = {columnsq2}")

Rows = 5 | Columns = 5


### a) verificar a dominância diagonal;

### Answer

It is diagonally dominant checked by the code bellow that sums the rows without the diagonals, in the output we can see that the value of the diagonal is bigger than the sum of it's row, so it is diagonally dominant.

In [338]:

for row in range(rowsq2):
    diagValue = abs(Aq2[row][row])
    rowValue = 0
    for column in range(columnsq2):
        if column != row:
            rowValue += abs(Aq2[row][column])
    print( f"A[{row}][{row}] Value = {diagValue}\nRow Sum Value: {rowValue}\n" )
    if diagValue <= rowValue:
        print( "Matrix Not Diagonally Dominant\n" )


A[0][0] Value = 10.0
Row Sum Value: 3.0

A[1][1] Value = 9.0
Row Sum Value: 5.0

A[2][2] Value = 10.0
Row Sum Value: 6.0

A[3][3] Value = 9.0
Row Sum Value: 5.0

A[4][4] Value = 8.0
Row Sum Value: 3.0



### b) resolver o sistema usando Gauss-Jacobi;

### Answer

Starting from $x^{(0)} = [0, 0, 0, 0, 0]$, with the relative norm tolerance 1e-10 and at most 100 iterations, the method converged in 34 iterations (`info = 0`). The solution found was [1.52310718 1.21132416 0.80842347 1.23885611 1.66076696].

For a system to converge using this method it is almost mandatory that the matrix has diagonal dominance, checked in item a), that it is square, which it is, and that the diagonal has no zeros, which also holds, so the method is suitable.

The Norma Relative is 1 in the first iteration because the starting vector is zero, so every component changes by 100% of its new value. After that it falls by a factor of about 2 at each iteration.

Table of iterations:

| Iteration | Norma Relative | V₁ | V₂ | V₃ | V₄ | V₅ |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 1.00000e+00 | 1.20000000 | 0.55555556 | 0.00000000 | 0.55555556 | 1.25000000 |
| 2 | 3.36400e-01 | 1.31111111 | 0.88395062 | 0.46722222 | 0.89506173 | 1.38888889 |
| 3 | 1.11724e-01 | 1.42351235 | 1.05019204 | 0.62580247 | 1.06624143 | 1.53216821 |
| 4 | 5.83478e-02 | 1.47261866 | 1.12943012 | 0.71885475 | 1.15179260 | 1.59478567 |
| 5 | 2.71103e-02 | 1.49777150 | 1.17052660 | 0.76298498 | 1.19519011 | 1.62780499 |
| 6 | 1.38161e-02 | 1.51040382 | 1.19074478 | 0.78570099 | 1.21690073 | 1.64417065 |
| 7 | 6.82964e-03 | 1.51671906 | 1.20101226 | 0.79698655 | 1.22783201 | 1.65243781 |
| 8 | 3.43961e-03 | 1.51990111 | 1.20613814 | 0.80268454 | 1.23331789 | 1.65658132 |
| 9 | 1.72121e-03 | 1.52149608 | 1.20872102 | 0.80553945 | 1.23607443 | 1.65866504 |
| 10 | 8.65063e-04 | 1.52229815 | 1.21001616 | 0.80697520 | 1.23745889 | 1.65971104 |
| 11 | 4.34111e-04 | 1.52270075 | 1.21066729 | 0.80769593 | 1.23815429 | 1.66023662 |
| 12 | 2.18082e-04 | 1.52290305 | 1.21099418 | 0.80805805 | 1.23850360 | 1.66050056 |
| 13 | 1.09515e-04 | 1.52300464 | 1.21115842 | 0.80823992 | 1.23867905 | 1.66063316 |
| 14 | 5.50104e-05 | 1.52305568 | 1.21124091 | 0.80833127 | 1.23876717 | 1.66069975 |
| 15 | 2.76296e-05 | 1.52308131 | 1.21128234 | 0.80837716 | 1.23881144 | 1.66073320 |
| 16 | 1.38782e-05 | 1.52309418 | 1.21130315 | 0.80840021 | 1.23883367 | 1.66075000 |
| 17 | 6.97080e-06 | 1.52310065 | 1.21131361 | 0.80841178 | 1.23884484 | 1.66075844 |
| 18 | 3.50138e-06 | 1.52310390 | 1.21131886 | 0.80841760 | 1.23885045 | 1.66076268 |
| 19 | 1.75870e-06 | 1.52310553 | 1.21132149 | 0.80842052 | 1.23885327 | 1.66076481 |
| 20 | 8.83381e-07 | 1.52310635 | 1.21132282 | 0.80842199 | 1.23885468 | 1.66076588 |
| 21 | 4.43714e-07 | 1.52310676 | 1.21132348 | 0.80842272 | 1.23885540 | 1.66076642 |
| 22 | 2.22873e-07 | 1.52310697 | 1.21132382 | 0.80842309 | 1.23885575 | 1.66076669 |
| 23 | 1.11947e-07 | 1.52310707 | 1.21132399 | 0.80842328 | 1.23885593 | 1.66076682 |
| 24 | 5.62300e-08 | 1.52310713 | 1.21132407 | 0.80842337 | 1.23885602 | 1.66076689 |
| 25 | 2.82438e-08 | 1.52310715 | 1.21132411 | 0.80842342 | 1.23885607 | 1.66076693 |
| 26 | 1.41866e-08 | 1.52310716 | 1.21132413 | 0.80842344 | 1.23885609 | 1.66076694 |
| 27 | 7.12581e-09 | 1.52310717 | 1.21132415 | 0.80842346 | 1.23885610 | 1.66076695 |
| 28 | 3.57923e-09 | 1.52310717 | 1.21132415 | 0.80842346 | 1.23885611 | 1.66076696 |
| 29 | 1.79781e-09 | 1.52310718 | 1.21132415 | 0.80842346 | 1.23885611 | 1.66076696 |
| 30 | 9.03025e-10 | 1.52310718 | 1.21132415 | 0.80842347 | 1.23885611 | 1.66076696 |
| 31 | 4.53581e-10 | 1.52310718 | 1.21132416 | 0.80842347 | 1.23885611 | 1.66076696 |
| 32 | 2.27830e-10 | 1.52310718 | 1.21132416 | 0.80842347 | 1.23885611 | 1.66076696 |
| 33 | 1.14437e-10 | 1.52310718 | 1.21132416 | 0.80842347 | 1.23885611 | 1.66076696 |
| 34 | 5.74806e-11 | 1.52310718 | 1.21132416 | 0.80842347 | 1.23885611 | 1.66076696 |

In [339]:
vectorSolutionJacobi, iterations, info = GaussJacobi(Aq2, bq2, 100, 1e-10, xo=xoq2, usingStartArray=True)

print(f"Info: {info}\n")
print(f"Solution: {vectorSolutionJacobi}\n")

Iteration: 1, Norma Relative: 1.00000e+00, Vector Solution: [1.2        0.55555556 0.         0.55555556 1.25      ]
Iteration: 2, Norma Relative: 3.36400e-01, Vector Solution: [1.31111111 0.88395062 0.46722222 0.89506173 1.38888889]
Iteration: 3, Norma Relative: 1.11724e-01, Vector Solution: [1.42351235 1.05019204 0.62580247 1.06624143 1.53216821]
Iteration: 4, Norma Relative: 5.83478e-02, Vector Solution: [1.47261866 1.12943012 0.71885475 1.1517926  1.59478567]
Iteration: 5, Norma Relative: 2.71103e-02, Vector Solution: [1.4977715  1.1705266  0.76298498 1.19519011 1.62780499]
Iteration: 6, Norma Relative: 1.38161e-02, Vector Solution: [1.51040382 1.19074478 0.78570099 1.21690073 1.64417065]
Iteration: 7, Norma Relative: 6.82964e-03, Vector Solution: [1.51671906 1.20101226 0.79698655 1.22783201 1.65243781]
Iteration: 8, Norma Relative: 3.43961e-03, Vector Solution: [1.51990111 1.20613814 0.80268454 1.23331789 1.65658132]
Iteration: 9, Norma Relative: 1.72121e-03, Vector Solution: [1.5

### c) resolver o sistema usando Gauss-Seidel;

### Answer

With the same starting vector, tolerance and limit, the method converged in 19 iterations (`info = 0`). The solution found was [1.52310718 1.21132416 0.80842347 1.23885611 1.66076696].

The requirements are the same as for Gauss-Jacobi, and they are all met. Seidel uses each new component as soon as it is computed, so it needs fewer iterations than Jacobi for the same tolerance (19 against 34).

Table of iterations:

| Iteration | Norma Relative | V₁ | V₂ | V₃ | V₄ | V₅ |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 1.00000e+00 | 1.20000000 | 0.82222222 | 0.28444444 | 0.71012346 | 1.46308642 |
| 2 | 2.61995e-01 | 1.39288889 | 1.00719890 | 0.62906200 | 1.13238842 | 1.61172985 |
| 3 | 8.45689e-02 | 1.46434598 | 1.14657826 | 0.76340092 | 1.21075998 | 1.64811511 |
| 4 | 2.88044e-02 | 1.50565574 | 1.19431926 | 0.79639293 | 1.23148170 | 1.65741954 |
| 5 | 7.73994e-03 | 1.51850314 | 1.20680821 | 0.80525025 | 1.23690531 | 1.65988261 |
| 6 | 2.03761e-03 | 1.52188667 | 1.21013102 | 0.80758419 | 1.23834051 | 1.66053315 |
| 7 | 5.40708e-04 | 1.52278462 | 1.21100868 | 0.80820162 | 1.23871980 | 1.66070515 |
| 8 | 1.42873e-04 | 1.52302190 | 1.21124076 | 0.80836482 | 1.23882008 | 1.66075062 |
| 9 | 3.77750e-05 | 1.52308463 | 1.21130211 | 0.80840796 | 1.23884659 | 1.66076264 |
| 10 | 9.98605e-06 | 1.52310122 | 1.21131833 | 0.80841937 | 1.23885359 | 1.66076582 |
| 11 | 2.63995e-06 | 1.52310560 | 1.21132262 | 0.80842238 | 1.23885545 | 1.66076666 |
| 12 | 6.97904e-07 | 1.52310676 | 1.21132375 | 0.80842318 | 1.23885594 | 1.66076688 |
| 13 | 1.84500e-07 | 1.52310707 | 1.21132405 | 0.80842339 | 1.23885607 | 1.66076694 |
| 14 | 4.87748e-08 | 1.52310715 | 1.21132413 | 0.80842345 | 1.23885610 | 1.66076696 |
| 15 | 1.28942e-08 | 1.52310717 | 1.21132415 | 0.80842346 | 1.23885611 | 1.66076696 |
| 16 | 3.40876e-09 | 1.52310718 | 1.21132415 | 0.80842347 | 1.23885611 | 1.66076696 |
| 17 | 9.01148e-10 | 1.52310718 | 1.21132416 | 0.80842347 | 1.23885611 | 1.66076696 |
| 18 | 2.38230e-10 | 1.52310718 | 1.21132416 | 0.80842347 | 1.23885611 | 1.66076696 |
| 19 | 6.29791e-11 | 1.52310718 | 1.21132416 | 0.80842347 | 1.23885611 | 1.66076696 |

In [355]:
vectorSolutionSeidel, iterations, info = GaussSeidel(Aq2, bq2, 100, 1e-10, xo=xoq2, usingStartArray=True)

print(f"Info: {info}\n")
print(f"Solution: {vectorSolutionSeidel}\n")

Iteration: 1, Norma Relative: 1.00000e+00, Vector Solution: [1.2        0.82222222 0.28444444 0.71012346 1.46308642]
Iteration: 2, Norma Relative: 2.61995e-01, Vector Solution: [1.39288889 1.0071989  0.629062   1.13238842 1.61172985]
Iteration: 3, Norma Relative: 8.45689e-02, Vector Solution: [1.46434598 1.14657826 0.76340092 1.21075998 1.64811511]
Iteration: 4, Norma Relative: 2.88044e-02, Vector Solution: [1.50565574 1.19431926 0.79639293 1.2314817  1.65741954]
Iteration: 5, Norma Relative: 7.73994e-03, Vector Solution: [1.51850314 1.20680821 0.80525025 1.23690531 1.65988261]
Iteration: 6, Norma Relative: 2.03761e-03, Vector Solution: [1.52188667 1.21013102 0.80758419 1.23834051 1.66053315]
Iteration: 7, Norma Relative: 5.40708e-04, Vector Solution: [1.52278462 1.21100868 0.80820162 1.2387198  1.66070515]
Iteration: 8, Norma Relative: 1.42873e-04, Vector Solution: [1.5230219  1.21124076 0.80836482 1.23882008 1.66075062]
Iteration: 9, Norma Relative: 3.77750e-05, Vector Solution: [1.5

### d) utilizar simultaneamente
$$
\frac{\lVert x^{(k+1)} - x^{(k)} \rVert_\infty}{\max\!\left(\lVert x^{(k+1)} \rVert_\infty,\, 1\right)} < 10^{-6}
\quad \text{e} \quad
\lVert b - A x^{(k+1)} \rVert_\infty < 10^{-8};
$$

### Answer

Now the iteration only stops when the relative step and the residue are below their tolerances at the same time. The solutions are the same as before:

Jacobi = [1.52310718 1.21132416 0.80842347 1.23885611 1.66076696] (30 iterations)

Seidel = [1.52310718 1.21132416 0.80842347 1.23885611 1.66076696] (17 iterations)

The residue criterion was the one that decided the stop in both methods. For Jacobi the relative step was already below 1e-6 at iteration 20, but the residue only went below 1e-8 at iteration 30. For Seidel the step was below 1e-6 at iteration 12 and the residue only at iteration 17. A small step alone does not guarantee that the equations are satisfied, which is why both criteria are used together.

Iteration table Jacobi:

| Iteration | Norma Relative | Residue | V₁ | V₂ | V₃ | V₄ | V₅ |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 1.00000e+00 | 4.67222e+00 | 1.20000000 | 0.55555556 | 0.00000000 | 0.55555556 | 1.25000000 |
| 2 | 3.36400e-01 | 1.58580e+00 | 1.31111111 | 0.88395062 | 0.46722222 | 0.89506173 | 1.38888889 |
| 3 | 1.11724e-01 | 9.30523e-01 | 1.42351235 | 1.05019204 | 0.62580247 | 1.06624143 | 1.53216821 |
| 4 | 5.83478e-02 | 4.41302e-01 | 1.47261866 | 1.12943012 | 0.71885475 | 1.15179260 | 1.59478567 |
| 5 | 2.71103e-02 | 2.27160e-01 | 1.49777150 | 1.17052660 | 0.76298498 | 1.19519011 | 1.62780499 |
| 6 | 1.38161e-02 | 1.12856e-01 | 1.51040382 | 1.19074478 | 0.78570099 | 1.21690073 | 1.64417065 |
| 7 | 6.82964e-03 | 5.69799e-02 | 1.51671906 | 1.20101226 | 0.79698655 | 1.22783201 | 1.65243781 |
| 8 | 3.43961e-03 | 2.85491e-02 | 1.51990111 | 1.20613814 | 0.80268454 | 1.23331789 | 1.65658132 |
| 9 | 1.72121e-03 | 1.43575e-02 | 1.52149608 | 1.20872102 | 0.80553945 | 1.23607443 | 1.65866504 |
| 10 | 8.65063e-04 | 7.20728e-03 | 1.52229815 | 1.21001616 | 0.80697520 | 1.23745889 | 1.65971104 |
| 11 | 4.34111e-04 | 3.62125e-03 | 1.52270075 | 1.21066729 | 0.80769593 | 1.23815429 | 1.66023662 |
| 12 | 2.18082e-04 | 1.81864e-03 | 1.52290305 | 1.21099418 | 0.80805805 | 1.23850360 | 1.66050056 |
| 13 | 1.09515e-04 | 9.13557e-04 | 1.52300464 | 1.21115842 | 0.80823992 | 1.23867905 | 1.66063316 |
| 14 | 5.50104e-05 | 4.58854e-04 | 1.52305568 | 1.21124091 | 0.80833127 | 1.23876717 | 1.66069975 |
| 15 | 2.76296e-05 | 2.30482e-04 | 1.52308131 | 1.21128234 | 0.80837716 | 1.23881144 | 1.66073320 |
| 16 | 1.38782e-05 | 1.15768e-04 | 1.52309418 | 1.21130315 | 0.80840021 | 1.23883367 | 1.66075000 |
| 17 | 6.97080e-06 | 5.81495e-05 | 1.52310065 | 1.21131361 | 0.80841178 | 1.23884484 | 1.66075844 |
| 18 | 3.50138e-06 | 2.92079e-05 | 1.52310390 | 1.21131886 | 0.80841760 | 1.23885045 | 1.66076268 |
| 19 | 1.75870e-06 | 1.46709e-05 | 1.52310553 | 1.21132149 | 0.80842052 | 1.23885327 | 1.66076481 |
| 20 | 8.83381e-07 | 7.36905e-06 | 1.52310635 | 1.21132282 | 0.80842199 | 1.23885468 | 1.66076588 |
| 21 | 4.43714e-07 | 3.70141e-06 | 1.52310676 | 1.21132348 | 0.80842272 | 1.23885540 | 1.66076642 |
| 22 | 2.22873e-07 | 1.85918e-06 | 1.52310697 | 1.21132382 | 0.80842309 | 1.23885575 | 1.66076669 |
| 23 | 1.11947e-07 | 9.33850e-07 | 1.52310707 | 1.21132399 | 0.80842328 | 1.23885593 | 1.66076682 |
| 24 | 5.62300e-08 | 4.69064e-07 | 1.52310713 | 1.21132407 | 0.80842337 | 1.23885602 | 1.66076689 |
| 25 | 2.82438e-08 | 2.35607e-07 | 1.52310715 | 1.21132411 | 0.80842342 | 1.23885607 | 1.66076693 |
| 26 | 1.41866e-08 | 1.18343e-07 | 1.52310716 | 1.21132413 | 0.80842344 | 1.23885609 | 1.66076694 |
| 27 | 7.12581e-09 | 5.94426e-08 | 1.52310717 | 1.21132415 | 0.80842346 | 1.23885610 | 1.66076695 |
| 28 | 3.57923e-09 | 2.98575e-08 | 1.52310717 | 1.21132415 | 0.80842346 | 1.23885611 | 1.66076696 |
| 29 | 1.79781e-09 | 1.49971e-08 | 1.52310718 | 1.21132415 | 0.80842346 | 1.23885611 | 1.66076696 |
| 30 | 9.03025e-10 | 7.53293e-09 | 1.52310718 | 1.21132415 | 0.80842347 | 1.23885611 | 1.66076696 |

Iteration table Seidel:

| Iteration | Norma Relative | Residue | V₁ | V₂ | V₃ | V₄ | V₅ |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 1.00000e+00 | 2.92617e+00 | 1.20000000 | 0.82222222 | 0.28444444 | 0.71012346 | 1.46308642 |
| 2 | 2.61995e-01 | 1.11150e+00 | 1.39288889 | 1.00719890 | 0.62906200 | 1.13238842 | 1.61172985 |
| 3 | 8.45689e-02 | 4.13098e-01 | 1.46434598 | 1.14657826 | 0.76340092 | 1.21075998 | 1.64811511 |
| 4 | 2.88044e-02 | 1.28474e-01 | 1.50565574 | 1.19431926 | 0.79639293 | 1.23148170 | 1.65741954 |
| 5 | 7.73994e-03 | 3.38352e-02 | 1.51850314 | 1.20680821 | 0.80525025 | 1.23690531 | 1.65988261 |
| 6 | 2.03761e-03 | 8.97956e-03 | 1.52188667 | 1.21013102 | 0.80758419 | 1.23834051 | 1.66053315 |
| 7 | 5.40708e-04 | 2.37276e-03 | 1.52278462 | 1.21100868 | 0.80820162 | 1.23871980 | 1.66070515 |
| 8 | 1.42873e-04 | 6.27354e-04 | 1.52302190 | 1.21124076 | 0.80836482 | 1.23882008 | 1.66075062 |
| 9 | 3.77750e-05 | 1.65845e-04 | 1.52308463 | 1.21130211 | 0.80840796 | 1.23884659 | 1.66076264 |
| 10 | 9.98605e-06 | 4.38434e-05 | 1.52310122 | 1.21131833 | 0.80841937 | 1.23885359 | 1.66076582 |
| 11 | 2.63995e-06 | 1.15906e-05 | 1.52310560 | 1.21132262 | 0.80842238 | 1.23885545 | 1.66076666 |
| 12 | 6.97904e-07 | 3.06411e-06 | 1.52310676 | 1.21132375 | 0.80842318 | 1.23885594 | 1.66076688 |
| 13 | 1.84500e-07 | 8.10037e-07 | 1.52310707 | 1.21132405 | 0.80842339 | 1.23885607 | 1.66076694 |
| 14 | 4.87748e-08 | 2.14143e-07 | 1.52310715 | 1.21132413 | 0.80842345 | 1.23885610 | 1.66076696 |
| 15 | 1.28942e-08 | 5.66115e-08 | 1.52310717 | 1.21132415 | 0.80842346 | 1.23885611 | 1.66076696 |
| 16 | 3.40876e-09 | 1.49660e-08 | 1.52310718 | 1.21132415 | 0.80842347 | 1.23885611 | 1.66076696 |
| 17 | 9.01148e-10 | 3.95645e-09 | 1.52310718 | 1.21132416 | 0.80842347 | 1.23885611 | 1.66076696 |

#### Modifed Methods

In [341]:
def GaussJacobiModified(A: np.array, b: np.array, maxIterations: int, precisionRelative: float, precisionResidue: float, xo:np.array = None, usingStartArray:bool = False) -> np.array:
    matrixSize = A.shape[0]
    v = np.zeros(matrixSize, dtype=float)
    vectorSolution = np.zeros(matrixSize, dtype=float)
    iterations = 0
    info = 0


    if A.shape[1] != matrixSize:
        raise ValueError("Matrix A must be square.")

    if not usingStartArray:
        for i in range(matrixSize):
            vectorSolution[i] = b[i] / A[i, i]
    else:
        vectorSolution = xo.copy()

    while True:
        iterations += 1
        for i in range(matrixSize):
            sumAx = 0.0
            for j in range(matrixSize):
                if i != j:
                    sumAx += A[i, j] * vectorSolution[j]
            v[i] = (b[i] - sumAx) / A[i, i]
        normaNum = 0.0
        normoDen = 0.0
        vectorSolutionPrev = vectorSolution.copy()
        for i in range(matrixSize):
            t = abs(v[i] - vectorSolution[i])
            if t > normaNum:
                normaNum = t
            if abs(v[i]) > normoDen:
                normoDen = abs(v[i])
            vectorSolution[i] = v[i]

        normaRel = np.linalg.norm(vectorSolution - vectorSolutionPrev, np.inf) / max(np.linalg.norm(vectorSolution, np.inf), 1)
        residue = np.linalg.norm(b - A @ vectorSolution, np.inf)
        print(f"Iteration: {iterations}, Norma Relative: {normaRel:.5e}, Residue: {residue:.5e}, Vector Solution: {vectorSolution}")

        if (iterations >= maxIterations) or ((normaRel <= precisionRelative) and (residue <= precisionResidue)):
            if iterations >= maxIterations:
                info = 1
            else:
                info = 0
            break
    
    return vectorSolution, iterations, info

def GaussSeidelModified(A: np.array, b: np.array, maxIterations: int, precisionRelative: float, precisionResidue:float, xo:np.array = None, usingStartArray:bool = False) -> np.array:
    matrixSize = A.shape[0]
    v = np.zeros(matrixSize, dtype=float)
    vectorSolution = np.zeros(matrixSize, dtype=float)
    iterations = 0
    info = 0


    if A.shape[1] != matrixSize:
        raise ValueError("Matrix A must be square.")

    if not usingStartArray:
        for i in range(matrixSize):
            vectorSolution[i] = b[i] / A[i, i]
    else:
        vectorSolution = xo.copy()

    while True:
        iterations += 1
        normaNum = 0.0
        normoDen = 0.0

        vectorSolutionPrev = vectorSolution.copy()

        for i in range(matrixSize):
            sumAx = 0.0
            for j in range(matrixSize):
                if i != j:
                    sumAx += A[i, j] * vectorSolution[j]
            v[i] = vectorSolution[i]
            vectorSolution[i] = (b[i] - sumAx) / A[i, i]
            t = abs(v[i] - vectorSolution[i])
            if t > normaNum:
                normaNum = t
            if abs(vectorSolution[i]) > normoDen:
                normoDen = abs(vectorSolution[i])
        
        normaRel = np.linalg.norm(vectorSolution - vectorSolutionPrev, np.inf) / max(np.linalg.norm(vectorSolution, np.inf), 1)
        residue = np.linalg.norm(b - A @ vectorSolution, np.inf)

        print(f"Iteration: {iterations}, Norma Relative: {normaRel:.5e}, Residue: {residue:.5e}, Vector Solution: {vectorSolution}")

        if (iterations >= maxIterations) or ((normaRel <= precisionRelative) and (residue <= precisionResidue)):
            if iterations >= maxIterations:
                info = 1
            else:
                info = 0
            break
    return vectorSolution, iterations, info

In [342]:
print("---Jacobi---\n")

vectorSolutionJacobi, iterations, info = GaussJacobiModified(Aq2, bq2, 100, 1e-6, 1e-8, xo=xoq2, usingStartArray=True)

print(f"Info: {info}\n")
print(f"Solution: {vectorSolutionJacobi}\n")

print("---Seidel---\n")


vectorSolutionSeidel, iterations, info = GaussSeidelModified(Aq2, bq2, 100, 1e-6, 1e-8, xo=xoq2, usingStartArray=True)

print(f"Info: {info}\n")
print(f"Solution: {vectorSolutionSeidel}\n")

---Jacobi---

Iteration: 1, Norma Relative: 1.00000e+00, Residue: 4.67222e+00, Vector Solution: [1.2        0.55555556 0.         0.55555556 1.25      ]
Iteration: 2, Norma Relative: 3.36400e-01, Residue: 1.58580e+00, Vector Solution: [1.31111111 0.88395062 0.46722222 0.89506173 1.38888889]
Iteration: 3, Norma Relative: 1.11724e-01, Residue: 9.30523e-01, Vector Solution: [1.42351235 1.05019204 0.62580247 1.06624143 1.53216821]
Iteration: 4, Norma Relative: 5.83478e-02, Residue: 4.41302e-01, Vector Solution: [1.47261866 1.12943012 0.71885475 1.1517926  1.59478567]
Iteration: 5, Norma Relative: 2.71103e-02, Residue: 2.27160e-01, Vector Solution: [1.4977715  1.1705266  0.76298498 1.19519011 1.62780499]
Iteration: 6, Norma Relative: 1.38161e-02, Residue: 1.12856e-01, Vector Solution: [1.51040382 1.19074478 0.78570099 1.21690073 1.64417065]
Iteration: 7, Norma Relative: 6.82964e-03, Residue: 5.69799e-02, Vector Solution: [1.51671906 1.20101226 0.79698655 1.22783201 1.65243781]
Iteration: 8,

### e) comparar as soluções.

Both methods found the same solution, [1.52310718 1.21132416 0.80842347 1.23885611 1.66076696], which also agrees with the direct solution of `np.linalg.solve(A, b)` to the 8 printed decimals (at most 1 unit in the last digit, within the residue tolerance of 1e-8).

| Method | Iterations (tolerance 1e-10) | Iterations (step < 1e-6 and residue < 1e-8) | Final residue |
|---|---:|---:|---:|
| Gauss-Jacobi | 34 | 30 | 7.53293e-09 |
| Gauss-Seidel | 19 | 17 | 3.95645e-09 |

Gauss-Seidel was faster in both comparisons, using 17 iterations against 30 with the combined criteria. This agrees with the spectral radius of the iteration matrices, which controls the convergence rate: $\rho_J = 0.5023$ for Jacobi and $\rho_{GS} = 0.2644$ for Seidel. Both are smaller than 1, so both converge, and Seidel gains about $\ln(0.2644)/\ln(0.5023) \approx 1.9$ times more correct digits per iteration, close to the ratio of iterations observed (30/17 = 1.76).

The cost of one iteration is about the same for both, and Seidel does not need the second vector to store the new values, so it is the better choice for this system. Jacobi has the advantage that its components are independent and can be computed in parallel.

## Questão 3 – Tubulações em paralelo

Duas tubulações em paralelo transportam uma vazão total $Q = 0{,}120\ \text{m}^3/\text{s}$. As perdas de carga são dadas por $h = K_1 q_1^2$ e $h = K_2 q_2^2$ com $K_1 = 1200$, $K_2 = 2500$.

O sistema resultante é

$$
\begin{aligned}
F_1(q_1, q_2, h) &= q_1 + q_2 - 0{,}120 = 0, \\
F_2(q_1, q_2, h) &= h - 1200\,q_1^2 = 0, \\
F_3(q_1, q_2, h) &= h - 2500\,q_2^2 = 0.
\end{aligned}
$$

Use

$$x^{(0)} = \begin{bmatrix} 0{,}060 \\ 0{,}060 \\ 5{,}0 \end{bmatrix}.$$

In [343]:
xoq3 = np.array([0.060, 0.060, 5.0 ])

### a) determinar analiticamente a Jacobiana;

### Answer

The Jacobian is the matrix of the partial derivatives of each $F_i$ with respect to $(q_1, q_2, h)$, obtained with `sympy`:

$$
J(q_1, q_2, h) = \begin{bmatrix}
1 & 1 & 0 \\
-2400\,q_1 & 0 & 1 \\
0 & -5000\,q_2 & 1
\end{bmatrix}
$$

In [344]:
q1, q2, h = sp.symbols('q1 q2 h')

Fq3temp = sp.Matrix([ q1 + q2 - 0.120, h - 1200 * q1**2, h - 2500 * q2**2 ])

Jq3temp = Fq3temp.jacobian([q1, q2, h])
Jq3temp

Matrix([
[       1,        1, 0],
[-2400*q1,        0, 1],
[       0, -5000*q2, 1]])

In [345]:
Fq3temp = sp.lambdify([q1, q2, h], Fq3temp, 'numpy')
Jq3temp = sp.lambdify([q1, q2, h], Jq3temp, 'numpy')

Fq3 = lambda x: np.array(Fq3temp(*x), dtype=float).flatten()
Jq3 = lambda x: np.array(Jq3temp(*x), dtype=float)

### b) resolver o sistema não linear usando Newton;

### Answer

Starting from $x^{(0)} = [0.060,\ 0.060,\ 5.0]$ with tolerance 1e-10, Newton converged in 4 iterations (`info = 0`) to

$$q_1 = 0.07088762\ \text{m}^3/\text{s}, \qquad q_2 = 0.04911238\ \text{m}^3/\text{s}, \qquad h = 6.03006522$$

| Iteration | q₁ | q₂ | h | $\lVert F \rVert_\infty$ before the step |
|---:|---:|---:|---:|---:|
| 1 | 0.07054054 | 0.04945946 | 5.83783784 | 4.00000e+00 |
| 2 | 0.07088724 | 0.04911276 | 6.02985703 | 2.77757e-01 |
| 3 | 0.07088762 | 0.04911238 | 6.03006522 | 3.00505e-04 |
| 4 | 0.07088762 | 0.04911238 | 6.03006522 | 3.53268e-10 |


In [346]:
vectorSolutionNewton, iterations, info = Newton(Fq3, Jq3, xoq3, precision=1e-10, maxIterations=100)

print(f"Info = {info}\n")
print(f"Solution = {vectorSolutionNewton}\n")

Iteration: 1, Vector Solution: [0.07054054 0.04945946 5.83783784], ||F(x)||_inf: 4.00000e+00
Iteration: 2, Vector Solution: [0.07088724 0.04911276 6.02985703], ||F(x)||_inf: 2.77757e-01
Iteration: 3, Vector Solution: [0.07088762 0.04911238 6.03006522], ||F(x)||_inf: 3.00505e-04
Iteration: 4, Vector Solution: [0.07088762 0.04911238 6.03006522], ||F(x)||_inf: 3.53268e-10
Info = 0

Solution = [0.07088762 0.04911238 6.03006522]



### c) para cada iteração resolver $J(x^k)\,\Delta x = -F(x^k)$;

### Answer

At each iteration the linear system $J(x^k)\,\Delta x = -F(x^k)$ is solved with `np.linalg.solve`, and the guess is updated with $x^{(k+1)} = x^{(k)} + \Delta x$.

| Iteration | Δq₁ | Δq₂ | Δh | q₁ | q₂ | h |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 1.05405e-02 | -1.05405e-02 | 8.37838e-01 | 0.07054054 | 0.04945946 | 5.83783784 |
| 2 | 3.46700e-04 | -3.46700e-04 | 1.92019e-01 | 0.07088724 | 0.04911276 | 6.02985703 |
| 3 | 3.75908e-07 | -3.75908e-07 | 2.08195e-04 | 0.07088762 | 0.04911238 | 6.03006522 |
| 4 | 4.41914e-13 | -4.41914e-13 | 2.44750e-10 | 0.07088762 | 0.04911238 | 6.03006522 |


In [347]:
def NewtonQ3Modified( F: callable , J: callable , x0: np.array , precision: float = 1e-10 , maxIterations: int = 100 ) -> tuple:
    vectorSolution = np.array(x0, dtype=float)
    iterations = 0
    info = 0
    while True:
        Jx = J(vectorSolution)
        Fx = F(vectorSolution)

        if np.linalg.norm(Fx, np.inf) <= precision:
            info = 0
            break

        try:
            s = np.linalg.solve(Jx, -Fx)
        except np.linalg.LinAlgError:
            info = 2
            break
    
        vectorSolution = vectorSolution + s

        iterations += 1
        
        print(f"Iteration: {iterations}, Delta x = {s},Vector Solution: {vectorSolution}, ||F(x)||_inf: {np.linalg.norm(Fx, np.inf):.5e}")

        if np.linalg.norm(s, np.inf) <= precision:
            info = 0
            break

        
        if iterations >= maxIterations:
            info = 1
            break
    return vectorSolution, iterations, info


vectorSolutionNewton, iterations, info = NewtonQ3Modified(Fq3, Jq3, xoq3, precision=1e-10, maxIterations=100)

print(f"Info = {info}\n")
print(f"Solution = {vectorSolutionNewton}\n")
        

Iteration: 1, Delta x = [ 0.01054054 -0.01054054  0.83783784],Vector Solution: [0.07054054 0.04945946 5.83783784], ||F(x)||_inf: 4.00000e+00
Iteration: 2, Delta x = [ 0.0003467  -0.0003467   0.19201919],Vector Solution: [0.07088724 0.04911276 6.02985703], ||F(x)||_inf: 2.77757e-01
Iteration: 3, Delta x = [ 3.75907887e-07 -3.75907887e-07  2.08195137e-04],Vector Solution: [0.07088762 0.04911238 6.03006522], ||F(x)||_inf: 3.00505e-04
Iteration: 4, Delta x = [ 4.41913918e-13 -4.41913918e-13  2.44750411e-10],Vector Solution: [0.07088762 0.04911238 6.03006522], ||F(x)||_inf: 3.53268e-10
Info = 0

Solution = [0.07088762 0.04911238 6.03006522]



### d) utilizar
$$
\frac{\lVert \Delta x \rVert_\infty}{\max\!\left(\lVert x^{(k+1)} \rVert_\infty,\, 1\right)} < 10^{-6}
\quad \text{ou} \quad
\lVert F(x^{k+1}) \rVert_\infty < 10^{-8}.
$$

### Answer

The iteration stops when at least one of the two criteria is satisfied (the "ou" of the statement). With them Newton stopped after 3 iterations (`info = 0`), with the same solution, $x = [0.07088762,\ 0.04911238,\ 6.03006522]$.

| Iteration | Relative step | $\lVert F(x^{(k+1)}) \rVert_\infty$ | Step < 1e-6 | Residual < 1e-8 |
|---:|---:|---:|:---:|:---:|
| 1 | 1.43519e-01 | 2.77757e-01 | no | no |
| 2 | 3.18447e-02 | 3.00505e-04 | no | no |
| 3 | 3.45262e-05 | 3.53268e-10 | no | yes |

In iteration 3 the relative step (3.5e-05) was still above 1e-6, but the residual (3.5e-10) was already below 1e-8, so the residual criterion stopped the method. Since $\lVert x \rVert_\infty \approx 6$ here (the largest unknown is $h$), the denominator makes the step relative to the size of $h$.


In [348]:
def NewtonQ3ModifiedV2( F: callable , J: callable , x0: np.array , precision1: float = 1e-10, precision2: float = 1e-10 , maxIterations: int = 100 ) -> tuple:
    vectorSolution = np.array(x0, dtype=float)
    iterations = 0
    info = 0
    while True:
        Jx = J(vectorSolution)
        Fx = F(vectorSolution)

        if np.linalg.norm(Fx, np.inf) <= precision2:
            info = 0
            break

        try:
            s = np.linalg.solve(Jx, -Fx)
        except np.linalg.LinAlgError:
            info = 2
            break
    
        vectorSolution = vectorSolution + s

        iterations += 1
        
        print(f"Iteration: {iterations}, Delta x = {s},Vector Solution: {vectorSolution}, ||F(x)||_inf: {np.linalg.norm(Fx, np.inf):.5e}")

        if (np.linalg.norm(s, np.inf) / max(np.linalg.norm(vectorSolution, np.inf), 1)) <= precision1:
            info = 0
            break

        
        if iterations >= maxIterations:
            info = 1
            break
    return vectorSolution, iterations, info


vectorSolutionNewton, iterations, info = NewtonQ3ModifiedV2(Fq3, Jq3, xoq3, precision1=1e-6, precision2=1e-8, maxIterations=100)

print(f"Info = {info}\n")
print(f"Solution = {vectorSolutionNewton}\n")
        

Iteration: 1, Delta x = [ 0.01054054 -0.01054054  0.83783784],Vector Solution: [0.07054054 0.04945946 5.83783784], ||F(x)||_inf: 4.00000e+00
Iteration: 2, Delta x = [ 0.0003467  -0.0003467   0.19201919],Vector Solution: [0.07088724 0.04911276 6.02985703], ||F(x)||_inf: 2.77757e-01
Iteration: 3, Delta x = [ 3.75907887e-07 -3.75907887e-07  2.08195137e-04],Vector Solution: [0.07088762 0.04911238 6.03006522], ||F(x)||_inf: 3.00505e-04
Info = 0

Solution = [0.07088762 0.04911238 6.03006522]



## Questão 4 – Fluxo de potência em duas barras

O fluxo de potência em um sistema de duas barras é dado por

$$
\begin{aligned}
F_1(V, \delta) &= \frac{V_1 V}{X}\sin\delta - P_L = 0, \\
F_2(V, \delta) &= \frac{V_1 V}{X}\cos\delta - \frac{V^2}{X} - Q_L = 0.
\end{aligned}
$$

Considere $V_1 = 1$ pu, $X = 0{,}5$ pu, $P_L = 0{,}6$ pu, $Q_L = 0{,}2$ pu.

Resolver o sistema não linear com

$$
x^{(0)} = \begin{bmatrix} 1{,}0 \\ 0{,}20 \end{bmatrix}
\quad \text{e depois com} \quad
x^{(0)} = \begin{bmatrix} 0{,}45 \\ 0{,}75 \end{bmatrix}.
$$

### a) determinar analiticamente a Jacobiana;

### Answer
 
With $V_1 = 1$ pu, $X = 0.5$ pu, $P_L = 0.6$ pu and $Q_L = 0.2$ pu, the factor $V_1 / X$ is 2 and the system becomes
 
$$
\begin{aligned}
F_1(V, \delta) &= 2V\sin\delta - 0.6 = 0, \\
F_2(V, \delta) &= 2V\cos\delta - 2V^2 - 0.2 = 0.
\end{aligned}
$$
 
The Jacobian, with respect to $(V, \delta)$, is
 
$$
J(V, \delta) = \begin{bmatrix}
\dfrac{\partial F_1}{\partial V} & \dfrac{\partial F_1}{\partial \delta} \\[2mm]
\dfrac{\partial F_2}{\partial V} & \dfrac{\partial F_2}{\partial \delta}
\end{bmatrix}
= \begin{bmatrix}
2\sin\delta & 2V\cos\delta \\
2\cos\delta - 4V & -2V\sin\delta
\end{bmatrix}
$$
 
It is the same matrix obtained with `sympy` in the code. In the general form, before replacing the data, it is
 
$$
J = \begin{bmatrix}
\dfrac{V_1}{X}\sin\delta & \dfrac{V_1 V}{X}\cos\delta \\[2mm]
\dfrac{V_1}{X}\cos\delta - \dfrac{2V}{X} & -\dfrac{V_1 V}{X}\sin\delta
\end{bmatrix}
$$

In [349]:
V, d = sp.symbols('V delta')

Fq4temp = sp.Matrix([ 2 * V * sp.sin(d) - 0.6, 2 * V * sp.cos(d) - 2 * V**2 - 0.2 ])

Jq4temp = Fq4temp.jacobian([V, d])
Jq4temp

Matrix([
[       2*sin(delta),  2*V*cos(delta)],
[-4*V + 2*cos(delta), -2*V*sin(delta)]])

### b) resolver o sistema não linear usando Newton Modificado para as duas estimativas iniciais;

### Answer
 
In the Modified Newton method the Jacobian is evaluated only once, at $x^{(0)}$, and the same matrix is used in all the iterations, $J(x^{(0)})\,s^{(k)} = -F(x^{(k)})$. The method was run with tolerance 1e-10 and at most 100 iterations.
 
| Initial guess | $J(x^{(0)})$ | Iterations | Info | $\lVert F \rVert_\infty$ at the end | Solution [V, δ] |
|---|---|---:|---:|---:|---|
| [1.0, 0.20] | [[0.3973, 1.9601], [-2.0399, -0.3973]] | 33 | 0 | 5.72130e-11 | [0.80308715, 0.38284225] |
| [0.45, 0.75] | [[1.3633, 0.6585], [-0.3366, -0.6135]] | 18 | 0 | 8.96685e-11 | [0.39376519, 0.86620353] |
 
Both runs converged, but to different solutions. The convergence is linear, because the Jacobian is never updated, so many more iterations are needed than in the full Newton method. With the `Newton` function, the full method needs only 5 iterations for the first guess and 4 for the second, and converges to the same two solutions.
 
Iterations for $x^{(0)} = [1.0,\ 0.20]$:
 
| Iteration | V | δ | $\lVert F \rVert_\infty$ before the step |
|---:|---:|---:|---:|
| 1 | 0.85660944 | 0.33245832 | 2.39867e-01 |
| 2 | 0.82780643 | 0.35914284 | 4.81515e-02 |
| 3 | 0.81544829 | 0.37088132 | 2.05448e-02 |
| 4 | 0.80947307 | 0.37663449 | 9.90271e-03 |
| 5 | 0.80643744 | 0.37957776 | 5.02280e-03 |
| 6 | 0.80485850 | 0.38111417 | 2.61036e-03 |
| 7 | 0.80402744 | 0.38192434 | 1.37333e-03 |
| 8 | 0.80358734 | 0.38235380 | 7.27111e-04 |
| 9 | 0.80335352 | 0.38258208 | 3.86252e-04 |
| 10 | 0.80322909 | 0.38270360 | 2.05544e-04 |
| 11 | 0.80316281 | 0.38276834 | 1.09482e-04 |
| 12 | 0.80312748 | 0.38280285 | 5.83439e-05 |
| 13 | 0.80310866 | 0.38282124 | 3.11003e-05 |
| 14 | 0.80309862 | 0.38283104 | 1.65803e-05 |
| 15 | 0.80309327 | 0.38283627 | 8.84006e-06 |
| 16 | 0.80309041 | 0.38283906 | 4.71340e-06 |
| 17 | 0.80308889 | 0.38284055 | 2.51317e-06 |
| 18 | 0.80308808 | 0.38284134 | 1.34003e-06 |
| 19 | 0.80308765 | 0.38284176 | 7.14516e-07 |
| 20 | 0.80308742 | 0.38284199 | 3.80986e-07 |
| 21 | 0.80308729 | 0.38284211 | 2.03146e-07 |
| 22 | 0.80308723 | 0.38284217 | 1.08319e-07 |
| 23 | 0.80308719 | 0.38284221 | 5.77570e-08 |
| 24 | 0.80308717 | 0.38284222 | 3.07967e-08 |
| 25 | 0.80308716 | 0.38284223 | 1.64211e-08 |
| 26 | 0.80308716 | 0.38284224 | 8.75592e-09 |
| 27 | 0.80308716 | 0.38284224 | 4.66875e-09 |
| 28 | 0.80308715 | 0.38284224 | 2.48943e-09 |
| 29 | 0.80308715 | 0.38284224 | 1.32739e-09 |
| 30 | 0.80308715 | 0.38284224 | 7.07779e-10 |
| 31 | 0.80308715 | 0.38284224 | 3.77396e-10 |
| 32 | 0.80308715 | 0.38284224 | 2.01231e-10 |
| 33 | 0.80308715 | 0.38284225 | 1.07299e-10 |
 
Iterations for $x^{(0)} = [0.45,\ 0.75]$:
 
| Iteration | V | δ | $\lVert F \rVert_\infty$ before the step |
|---:|---:|---:|---:|
| 1 | 0.37921258 | 0.87608275 | 5.35200e-02 |
| 2 | 0.39876436 | 0.86195281 | 1.73497e-02 |
| 3 | 0.39214770 | 0.86742595 | 5.41618e-03 |
| 4 | 0.39429226 | 0.86578662 | 1.84411e-03 |
| 5 | 0.39359346 | 0.86633715 | 5.90123e-04 |
| 6 | 0.39382111 | 0.86615977 | 1.93545e-04 |
| 7 | 0.39374698 | 0.86621775 | 6.28884e-05 |
| 8 | 0.39377112 | 0.86619889 | 2.04991e-05 |
| 9 | 0.39376326 | 0.86620504 | 6.67479e-06 |
| 10 | 0.39376582 | 0.86620304 | 2.17417e-06 |
| 11 | 0.39376499 | 0.86620369 | 7.08106e-07 |
| 12 | 0.39376526 | 0.86620348 | 2.30632e-07 |
| 13 | 0.39376517 | 0.86620354 | 7.51165e-08 |
| 14 | 0.39376520 | 0.86620352 | 2.44655e-08 |
| 15 | 0.39376519 | 0.86620353 | 7.96839e-09 |
| 16 | 0.39376519 | 0.86620353 | 2.59530e-09 |
| 17 | 0.39376519 | 0.86620353 | 8.45290e-10 |
| 18 | 0.39376519 | 0.86620353 | 2.75311e-10 |

In [350]:
Fq4temp = sp.lambdify([V, d], Fq4temp, 'numpy')
Jq4temp = sp.lambdify([V, d], Jq4temp, 'numpy')

Fq4 = lambda x: np.array(Fq4temp(*x), dtype=float).flatten()
Jq4 = lambda x: np.array(Jq4temp(*x), dtype=float)

xoq41 = np.array([1.0, 0.20], dtype=float)
xoq42 = np.array([0.45, 0.75], dtype=float)

In [351]:
print(f"For input = {xoq41}\n")

vectorSolutionQ4xo1, iterationsQ4xo1, infoQ4xo1 = ModifiedNewton(Fq4, Jq4, xoq41, precision=1e-10, maxIterations=100)

print(f"Info = {infoQ4xo1}\n")
print(f"Solution = {vectorSolutionQ4xo1}\n")

print(f"For input = {xoq42}\n")

vectorSolutionQ4xo2, iterationsQ4xo2, infoQ4xo2 = ModifiedNewton(Fq4, Jq4, xoq42, precision=1e-10, maxIterations=100)

print(f"Info = {infoQ4xo2}\n")
print(f"Solution = {vectorSolutionQ4xo2}\n")

For input = [1.  0.2]

Iteration: 1, Vector Solution: [0.85660944 0.33245832], ||F(x)||_inf: 2.39867e-01
Iteration: 2, Vector Solution: [0.82780643 0.35914284], ||F(x)||_inf: 4.81515e-02
Iteration: 3, Vector Solution: [0.81544829 0.37088132], ||F(x)||_inf: 2.05448e-02
Iteration: 4, Vector Solution: [0.80947307 0.37663449], ||F(x)||_inf: 9.90271e-03
Iteration: 5, Vector Solution: [0.80643744 0.37957776], ||F(x)||_inf: 5.02280e-03
Iteration: 6, Vector Solution: [0.8048585  0.38111417], ||F(x)||_inf: 2.61036e-03
Iteration: 7, Vector Solution: [0.80402744 0.38192434], ||F(x)||_inf: 1.37333e-03
Iteration: 8, Vector Solution: [0.80358734 0.3823538 ], ||F(x)||_inf: 7.27111e-04
Iteration: 9, Vector Solution: [0.80335352 0.38258208], ||F(x)||_inf: 3.86252e-04
Iteration: 10, Vector Solution: [0.80322909 0.3827036 ], ||F(x)||_inf: 2.05544e-04
Iteration: 11, Vector Solution: [0.80316281 0.38276834], ||F(x)||_inf: 1.09482e-04
Iteration: 12, Vector Solution: [0.80312748 0.38280285], ||F(x)||_inf: 5.

### c) determinar $V$ e $\delta$;

### Answer
 
| Initial guess | V (pu) | δ (rad) | δ (deg) |
|---|---:|---:|---:|
| [1.0, 0.20] | 0.80308715 | 0.38284225 | 21.9352 |
| [0.45, 0.75] | 0.39376519 | 0.86620353 | 49.6298 |
 
Both pairs satisfy the original equations: replacing them in the power flow expressions gives $P = 0.60000000$ pu and $Q = 0.20000000$ pu, equal to the load $P_L$ and $Q_L$.

In [352]:
V1 = 1.0
X = 0.5

for name, solution in (("x0 = [1.0, 0.20]", vectorSolutionQ4xo1), ("x0 = [0.45, 0.75]", vectorSolutionQ4xo2)):
    Vsol = solution[0]
    deltaSol = solution[1]
    print(f"{name}")
    print(f"V = {Vsol:.8f} pu")
    print(f"delta = {deltaSol:.8f} rad = {np.degrees(deltaSol):.4f} deg")
    print(f"P = {V1 * Vsol / X * np.sin(deltaSol):.8f} pu | Q = {V1 * Vsol / X * np.cos(deltaSol) - Vsol**2 / X:.8f} pu\n")

x0 = [1.0, 0.20]
V = 0.80308715 pu
delta = 0.38284225 rad = 21.9352 deg
P = 0.60000000 pu | Q = 0.20000000 pu

x0 = [0.45, 0.75]
V = 0.39376519 pu
delta = 0.86620353 rad = 49.6298 deg
P = 0.60000000 pu | Q = 0.20000000 pu



### d) comparar as duas raízes.

### Answer
 
The system has two different roots, and the initial guess decides which one the method finds:
 
| | Root 1 | Root 2 |
|---|---:|---:|
| Initial guess | [1.0, 0.20] | [0.45, 0.75] |
| V (pu) | 0.80308715 | 0.39376519 |
| δ (deg) | 21.9352 | 49.6298 |
| Iterations (Modified Newton) | 33 | 18 |
 
The difference between them is 0.4093 pu in voltage and 27.6946° in angle. No other root with $V > 0$ was found when scanning other starting points.
 
Root 1 is the usual operating point: the bus voltage is about 0.80 pu and the angle is smaller. Root 2 is the low-voltage solution of the power flow equations: the voltage is less than half of the nominal and the angle is much larger. Both deliver the same power, but with a lower voltage the current is about twice as large ($|S| / V$ is 0.788 pu against 1.606 pu), which means higher losses. In practice, this low-voltage root is normally not a viable operating point, because the voltage is far below the acceptable limits.
 
The practical conclusion is that for nonlinear systems the result depends on the initial guess, and the physical knowledge of the problem (here, a voltage close to 1 pu) is needed to choose a guess that leads to the meaningful root.

In [353]:
residualQ4xo1 = np.linalg.norm(Fq4(vectorSolutionQ4xo1), np.inf)
residualQ4xo2 = np.linalg.norm(Fq4(vectorSolutionQ4xo2), np.inf)

print(f"{'x0':<14}{'V (pu)':<14}{'delta (rad)':<14}{'delta (deg)':<14}{'Iterations':<12}{'Info':<6}{'||F||_inf'}")
print(f"{str(xoq41):<14}{vectorSolutionQ4xo1[0]:<14.8f}{vectorSolutionQ4xo1[1]:<14.8f}{np.degrees(vectorSolutionQ4xo1[1]):<14.4f}{iterationsQ4xo1:<12}{infoQ4xo1:<6}{residualQ4xo1:.5e}")
print(f"{str(xoq42):<14}{vectorSolutionQ4xo2[0]:<14.8f}{vectorSolutionQ4xo2[1]:<14.8f}{np.degrees(vectorSolutionQ4xo2[1]):<14.4f}{iterationsQ4xo2:<12}{infoQ4xo2:<6}{residualQ4xo2:.5e}")

print(f"\nDifference in V: {abs(vectorSolutionQ4xo1[0] - vectorSolutionQ4xo2[0]):.8f} pu")
print(f"Difference in delta: {np.degrees(abs(vectorSolutionQ4xo1[1] - vectorSolutionQ4xo2[1])):.4f} deg")
print(f"Same root? {np.allclose(vectorSolutionQ4xo1, vectorSolutionQ4xo2)}")

x0            V (pu)        delta (rad)   delta (deg)   Iterations  Info  ||F||_inf
[1.  0.2]     0.80308715    0.38284225    21.9352       33          0     5.72130e-11
[0.45 0.75]   0.39376519    0.86620353    49.6298       18          0     8.96685e-11

Difference in V: 0.40932196 pu
Difference in delta: 27.6946 deg
Same root? False
